In [7]:
import pandas as pd
import numpy as np
from faker import Faker
import random

In [10]:
fake = Faker()
random.seed(42)
np.random.seed(42)

N_NORMAL = 4800          # normal transactions
N_ANOMALIES = 200        # deliberately planted anomalies (~4% of population)

In [11]:
departments = ["Procurement", "Sales", "Finance", "Operations", "HR", "IT", "Marketing"]
gl_accounts = {
    "5000": "Raw Materials Expense",
    "5100": "Office Supplies",
    "5200": "Travel & Entertainment",
    "5300": "Professional Fees",
    "5400": "IT & Software",
    "5500": "Utilities",
    "6000": "Vendor Payments",
}
employees = [fake.name() for _ in range(40)]
vendors = [fake.company() for _ in range(60)]

In [12]:
def random_business_datetime(start="2024-01-01", end="2024-12-31", weekend_ok=False, odd_hour=False):
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    while True:
        d = start_ts + pd.Timedelta(days=random.randint(0, (end_ts - start_ts).days))
        if not weekend_ok and d.weekday() >= 5:
            continue
        if odd_hour:
            hour = random.choice([0, 1, 2, 3, 22, 23])
        else:
            hour = random.randint(9, 18)
        minute = random.randint(0, 59)
        return d.replace(hour=hour, minute=minute)

rows = []
entry_id = 100000


In [13]:
# ---------- 1. NORMAL TRANSACTIONS ----------
for _ in range(N_NORMAL):
    entry_id += 1
    dept = random.choice(departments)
    gl_code = random.choice(list(gl_accounts.keys()))
    vendor = random.choice(vendors)
    preparer = random.choice(employees)
    approver = random.choice([e for e in employees if e != preparer])  # proper segregation
    amount = round(np.random.gamma(shape=2.0, scale=8000) + 500, 2)     # realistic skewed amounts
    post_dt = random_business_datetime()
    po_number = f"PO{fake.unique.random_number(digits=6)}"
    invoice_number = f"INV{fake.unique.random_number(digits=6)}"

    rows.append({
        "entry_id": entry_id,
        "date": post_dt.date(),
        "posting_time": post_dt.time(),
        "posting_datetime": post_dt,
        "department": dept,
        "gl_account": gl_code,
        "account_description": gl_accounts[gl_code],
        "vendor": vendor,
        "amount": amount,
        "preparer": preparer,
        "approver": approver,
        "po_number": po_number,
        "invoice_number": invoice_number,
        "po_amount": amount,
        "invoice_amount": amount,
        "payment_amount": amount,
        "planted_anomaly_type": "none",
    })

In [14]:
# ---------- 2. PLANTED ANOMALIES (so we can later validate detection rate) ----------
anomaly_types = [
    "duplicate_payment", "duplicate_payment",       # weight duplicates a bit more
    "three_way_mismatch",
    "weekend_posting",
    "odd_hour_posting",
    "round_number",
    "segregation_violation",
    "unusual_amount",
]

for _ in range(N_ANOMALIES):
    entry_id += 1
    dept = random.choice(departments)
    gl_code = random.choice(list(gl_accounts.keys()))
    vendor = random.choice(vendors)
    preparer = random.choice(employees)
    a_type = random.choice(anomaly_types)

    amount = round(np.random.gamma(shape=2.0, scale=8000) + 500, 2)
    post_dt = random_business_datetime()
    po_number = f"PO{fake.unique.random_number(digits=6)}"
    invoice_number = f"INV{fake.unique.random_number(digits=6)}"
    approver = random.choice([e for e in employees if e != preparer])
    po_amount, invoice_amount, payment_amount = amount, amount, amount

    if a_type == "weekend_posting":
        post_dt = random_business_datetime(weekend_ok=True)
        while post_dt.weekday() < 5:
            post_dt = random_business_datetime(weekend_ok=True)

    elif a_type == "odd_hour_posting":
        post_dt = random_business_datetime(odd_hour=True)

    elif a_type == "round_number":
        amount = float(random.choice([10000, 25000, 50000, 100000, 75000]))
        po_amount = invoice_amount = payment_amount = amount

    elif a_type == "segregation_violation":
        approver = preparer  # same person prepares AND approves -- classic SoD red flag

    elif a_type == "three_way_mismatch":
        invoice_amount = round(amount * random.uniform(1.05, 1.25), 2)  # invoice inflated vs PO
        payment_amount = invoice_amount

    elif a_type == "unusual_amount":
        amount = round(amount * random.uniform(6, 12), 2)  # way bigger than typical
        po_amount = invoice_amount = payment_amount = amount

    rows.append({
        "entry_id": entry_id,
        "date": post_dt.date(),
        "posting_time": post_dt.time(),
        "posting_datetime": post_dt,
        "department": dept,
        "gl_account": gl_code,
        "account_description": gl_accounts[gl_code],
        "vendor": vendor,
        "amount": amount,
        "preparer": preparer,
        "approver": approver,
        "po_number": po_number,
        "invoice_number": invoice_number,
        "po_amount": po_amount,
        "invoice_amount": invoice_amount,
        "payment_amount": payment_amount,
        "planted_anomaly_type": a_type,
    })

df = pd.DataFrame(rows)

In [15]:
# ---------- 3. INJECT ACTUAL DUPLICATE PAYMENTS (needs to duplicate real rows) ----------
dup_candidates = df[df["planted_anomaly_type"] == "duplicate_payment"].copy()
dup_rows = []
for _, r in dup_candidates.iterrows():
    entry_id += 1
    dup = r.copy()
    dup["entry_id"] = entry_id
    # same vendor, same amount, same invoice number posted again a few days later
    dup["posting_datetime"] = r["posting_datetime"] + pd.Timedelta(days=random.randint(2, 10))
    dup["date"] = dup["posting_datetime"].date()
    dup["posting_time"] = dup["posting_datetime"].time()
    dup_rows.append(dup)

df = pd.concat([df, pd.DataFrame(dup_rows)], ignore_index=True)

In [17]:
df = df.sort_values("posting_datetime").reset_index(drop=True)
df.to_csv("journal_entries_raw.csv", index=False)

print(f"Generated {len(df)} journal entries")
print(f"Planted anomalies by type:\n{df['planted_anomaly_type'].value_counts()}")

Generated 5047 journal entries
Planted anomalies by type:
planted_anomaly_type
none                     4800
duplicate_payment          94
unusual_amount             34
segregation_violation      33
weekend_posting            25
round_number               23
odd_hour_posting           21
three_way_mismatch         17
Name: count, dtype: int64


In [16]:
df.head()

,entry_id,date,posting_time,posting_datetime,department,gl_account,account_description,vendor,amount,preparer,approver,po_number,invoice_number,po_amount,invoice_amount,payment_amount,planted_anomaly_type
0,100001,2024-04-24,11:47:00,2024-04-24 11:47:00,IT,5000,Raw Materials Expense,"Jefferson, Sullivan and Young",19649.44,Gary Stone,Lawrence Davis,PO521629,INV552608,19649.44,19649.44,19649.44,none
1,100002,2024-10-29,15:02:00,2024-10-29 15:02:00,Procurement,5500,Utilities,Pearson PLC,12455.72,Sergio Arroyo,Donald Ferguson,PO199604,INV689555,12455.72,12455.72,12455.72,none
2,100003,2024-11-04,09:35:00,2024-11-04 09:35:00,Procurement,5000,Raw Materials Expense,Young Inc,11558.27,James Dennis,Joy Hernandez,PO992192,INV920462,11558.27,11558.27,11558.27,none
3,100004,2024-04-22,16:37:00,2024-04-22 16:37:00,Sales,5500,Utilities,Willis Group,11558.42,Sergio Arroyo,Michael Macdonald,PO900182,INV951265,11558.42,11558.42,11558.42,none
4,100005,2024-12-23,15:21:00,2024-12-23 15:21:00,Finance,6000,Vendor Payments,"Silva, Jones and Oliver",37697.72,Autumn Reed,Patricia Lopez,PO18481,INV811158,37697.72,37697.72,37697.72,none
